# Electrochemical Sensor Data Analysis

This notebook demonstrates a reproducible workflow for cyclic voltammetry (CV) and electrochemical impedance spectroscopy (EIS).

**Important:** all included measurements are synthetic demonstration data, not real laboratory results.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
from scipy.signal import find_peaks

ROOT = Path.cwd()
if not (ROOT / "data").exists() and (ROOT.parent / "data").exists():
    ROOT = ROOT.parent
DATA = ROOT / "data"
OUT = ROOT / "outputs"
OUT.mkdir(exist_ok=True)
print("Project root:", ROOT)

## 1. Load and inspect the CV data

In [ ]:
cv = pd.read_csv(DATA / "sample_cv_data.csv")
display(cv.head())
print(cv.info())
print(cv.groupby("scan_direction")[["potential_V", "current_uA"]].agg(["count", "min", "max", "mean"]))

## 2. Plot cyclic voltammetry

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
for direction, group in cv.groupby("scan_direction", sort=False):
    ax.plot(group["potential_V"], group["current_uA"], label=direction.title())
ax.set(title="Cyclic Voltammetry — Synthetic Demonstration Data",
       xlabel="Potential (V)", ylabel="Current (µA)")
ax.grid(True, alpha=0.3)
ax.legend()
fig.tight_layout()
fig.savefig(OUT / "cv_plot.png", dpi=180)
plt.show()

## 3. Estimate peak candidates

This is a basic numerical extrema search, not a validated peak-analysis method. Real data may require background subtraction, smoothing, replicate checks, and chemistry-specific interpretation.

In [ ]:
peak_rows = []
for direction, group in cv.groupby("scan_direction", sort=False):
    # Sort ascending so the x-axis is consistent for peak detection.
    group = group.sort_values("potential_V")
    x = group["potential_V"].to_numpy()
    y = group["current_uA"].to_numpy()
    if direction == "forward":
        idxs, _ = find_peaks(y, prominence=max(y.std() * 0.8, 0.5))
        if len(idxs):
            idx = idxs[y[idxs].argmax()]
            peak_rows.append({"scan_direction": direction, "peak_type": "anodic_candidate",
                              "potential_V": x[idx], "current_uA": y[idx]})
    else:
        idxs, _ = find_peaks(-y, prominence=max(y.std() * 0.8, 0.5))
        if len(idxs):
            idx = idxs[(-y[idxs]).argmax()]
            peak_rows.append({"scan_direction": direction, "peak_type": "cathodic_candidate",
                              "potential_V": x[idx], "current_uA": y[idx]})
peak_summary = pd.DataFrame(peak_rows)
display(peak_summary)
peak_summary.to_csv(OUT / "peak_summary.csv", index=False)

## 4. Load and plot EIS data

In [ ]:
eis = pd.read_csv(DATA / "sample_eis_data.csv")
display(eis.head())
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(8, 7), sharex=True)
ax1.loglog(eis["frequency_Hz"], eis["z_magnitude_ohm"], marker="o", markersize=3)
ax1.set_ylabel("|Z| (Ω)")
ax1.set_title("EIS Bode Plot — Synthetic Demonstration Data")
ax1.grid(True, which="both", alpha=0.3)
ax2.semilogx(eis["frequency_Hz"], eis["phase_deg"], marker="o", markersize=3)
ax2.set_xlabel("Frequency (Hz)")
ax2.set_ylabel("Phase (degrees)")
ax2.grid(True, which="both", alpha=0.3)
fig.tight_layout()
fig.savefig(OUT / "eis_bode_plot.png", dpi=180)
plt.show()

## 5. Interpretation and limitations

- The CV plot shows current response as potential is swept in forward and reverse directions.
- Peak candidates are algorithmic estimates and do not, by themselves, establish analyte identity or sensor performance.
- The EIS Bode plot shows impedance magnitude and phase versus frequency.
- The included EIS data are generated from a simple illustrative RC response. They are not fitted experimental measurements.
- Do not report sensitivity, selectivity, limit of detection, or equivalent-circuit parameters as experimental findings from these synthetic files.